# Fine-tune a small VLM to classify medical bill pages**What this does:** teaches a 2-billion-parameter vision model to look at a bill pageand say what type of page it is — Bill Summary, Bill Detail, Pharmacy Bill, Lab Bill, or Other.**What this does NOT do:** it does not extract line items. That is a much harder task and50 pages is not enough data for it. We are doing the smaller task properly instead of thebigger task badly.**Before you start — Runtime → Change runtime type → T4 GPU.**---### How the notebook is organised1. Check the GPU2. Install libraries3. Upload your PDFs4. Turn PDF pages into images5. Label the pages by hand (about 15 minutes)6. Split into train and test7. Load the model8. **Measure the model BEFORE training** ← most people skip this. Do not skip it.9. Train with LoRA10. Measure again and compareStep 8 is the important one. If you do not know the score before training, you cannot saywhether training helped.

## 1. Check the GPU

In [ ]:
import torchprint("GPU available:", torch.cuda.is_available())if torch.cuda.is_available():    print("GPU name:", torch.cuda.get_device_name(0))    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9    print(f"GPU memory: {total_memory_gb:.1f} GB")    # A T4 is an older GPU. It cannot do bfloat16 maths, only float16.    # If we use the wrong one, training crashes with a confusing dtype error.    major_version = torch.cuda.get_device_capability()[0]    if major_version >= 8:        print("This GPU supports bfloat16 -> we will use bfloat16")    else:        print("This GPU is older (T4 or similar) -> we must use float16")else:    print("NO GPU. Go to Runtime -> Change runtime type -> T4 GPU")

## 2. Install librariesTakes about 2 minutes.

In [ ]:
!pip -q install "transformers>=4.49.0" "peft>=0.13.0" "bitsandbytes>=0.44.0" accelerate pymupdfprint("Done installing")

## 3. Upload your PDFsRun this cell, then choose all 15 `train_sample_*.pdf` files from your computer.

In [ ]:
import osos.makedirs("pdfs", exist_ok=True)from google.colab import filesuploaded = files.upload()          # a file picker will appearfor filename in uploaded:    os.rename(filename, f"pdfs/{filename}")print("Uploaded", len(os.listdir("pdfs")), "files")

## 4. Turn PDF pages into imagesThe model looks at pictures, not PDFs. So we convert every page into a PNG image.We name each image `documentname_pageN.png` so we always know which document it came from.That matters later when we split the data.

In [ ]:
import fitz          # PyMuPDF, for reading PDFsimport osos.makedirs("pages", exist_ok=True)all_pages = []       # we will fill this with information about every pagefor pdf_name in sorted(os.listdir("pdfs")):    if not pdf_name.endswith(".pdf"):        continue    document = fitz.open(f"pdfs/{pdf_name}")    doc_name = pdf_name.replace(".pdf", "")    for page_number in range(len(document)):        page = document[page_number]        # zoom=2 means render at roughly 144 DPI. Higher is sharper but uses more memory.        image = page.get_pixmap(matrix=fitz.Matrix(2, 2))        image_path = f"pages/{doc_name}_page{page_number + 1}.png"        image.save(image_path)        all_pages.append({            "image_path": image_path,            "document": doc_name,        # which PDF this page came from            "page_number": page_number + 1,        })    document.close()print(f"Made {len(all_pages)} page images from {len(os.listdir('pdfs'))} PDFs")

## 5. Label the pages by handThis is the part that takes your time — around 15 minutes for 50 pages. There is noshortcut worth taking here.You could ask Gemini to label them for you. But then, when you test your model, you wouldonly be measuring *"does my model agree with Gemini"* — not *"is my model correct"*. And yourmodel could never score better than Gemini, because Gemini's answers would be the answer key.So label them yourself.**Run the next cell.** It shows you each page image with a text box. Type the label and press Enter.

In [ ]:
from IPython.display import display, Image as ShowImage# The five labels. Type them exactly, or type the number shown.LABEL_OPTIONS = ["Bill Summary", "Bill Detail", "Pharmacy Bill", "Lab Bill", "Other"]print("What the labels mean:")print("  1 Bill Summary  - only category totals, no individual items")print("  2 Bill Detail   - a table of individual services with amounts")print("  3 Pharmacy Bill - a medicine / drug bill")print("  4 Lab Bill      - a pathology or diagnostic test bill")print("  5 Other         - anything else (implant invoice, cover page, etc.)")print()for page in all_pages:    if "label" in page:          # already labelled, skip it        continue    display(ShowImage(filename=page["image_path"], width=520))    print(f"{page['document']}  page {page['page_number']}")    answer = input("Label (type 1-5, or the full name): ").strip()    if answer in ["1", "2", "3", "4", "5"]:        page["label"] = LABEL_OPTIONS[int(answer) - 1]    elif answer in LABEL_OPTIONS:        page["label"] = answer    else:        print("  Not understood, marking as Other")        page["label"] = "Other"    print(f"  -> {page['label']}\n")print("Finished labelling all", len(all_pages), "pages")

### Save your labelsLabelling took real effort. Save it to a file so that if Colab disconnects you do not lose it.Download `my_labels.json` to your computer, and re-upload it next time instead of labelling again.

In [ ]:
import jsonwith open("my_labels.json", "w") as f:    json.dump(all_pages, f, indent=2)print("Saved to my_labels.json")# Count how many of each label we havefrom collections import Counterlabel_counts = Counter(page["label"] for page in all_pages)print("\nHow many pages of each type:")for label, count in label_counts.most_common():    print(f"  {label:<15} {count}")# The most common label is important. If 70% of pages are "Bill Detail", then a model# that always guesses "Bill Detail" gets 70% - without learning anything at all.# We must beat that number, otherwise our training did nothing useful.most_common_label, most_common_count = label_counts.most_common(1)[0]baseline_accuracy = most_common_count / len(all_pages)print(f"\nIf we always guessed '{most_common_label}' we would be right "      f"{baseline_accuracy:.0%} of the time.")print("Our trained model must beat this to be worth anything.")

## 6. Split into train and test**Important:** we split by *document*, not by page.Pages from the same bill share the same hospital template, the same printer, the same scanner.If page 1 of a bill is in training and page 2 is in testing, the model has effectively alreadyseen the test page. The score would look great and mean nothing.So all pages from one document go to the same side.

In [ ]:
import random# NOTE: train_sample_9 and train_sample_10 are actually the SAME 90-page bill,# split into two files. So we treat them as one document.SAME_DOCUMENT = {"train_sample_9": "bill_A", "train_sample_10": "bill_A"}def document_group(doc_name):    return SAME_DOCUMENT.get(doc_name, doc_name)# Get the list of unique documents and shuffle it with a fixed seed,# so we get the same split every time we run this.all_groups = sorted(set(document_group(p["document"]) for p in all_pages))random.Random(42).shuffle(all_groups)number_for_testing = max(1, round(len(all_groups) * 0.3))test_groups = set(all_groups[:number_for_testing])train_pages = [p for p in all_pages if document_group(p["document"]) not in test_groups]test_pages  = [p for p in all_pages if document_group(p["document"]) in test_groups]print(f"Training on {len(train_pages)} pages from {len(all_groups) - number_for_testing} documents")print(f"Testing on  {len(test_pages)} pages from {number_for_testing} documents")print()print("WARNING: with so few test pages, the accuracy number is rough.")print("If you get 80% on 15 pages, the true value could easily be 60% or 95%.")print("Say this honestly rather than quoting a precise-sounding number.")

## 7. Load the modelWe load **Qwen2-VL-2B** in 4-bit. "4-bit" means each number in the model is squeezed into4 bits instead of 16, so the model uses about a quarter of the memory. It fits easily on a T4.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfigimport torchMODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"# T4 cannot do bfloat16, so choose based on the GPU we actually have.use_bfloat16 = torch.cuda.get_device_capability()[0] >= 8compute_dtype = torch.bfloat16 if use_bfloat16 else torch.float16print("Using", "bfloat16" if use_bfloat16 else "float16")# Settings for loading the model in 4-bitquantization_settings = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_quant_type="nf4",    bnb_4bit_compute_dtype=compute_dtype,    bnb_4bit_use_double_quant=True,)model = Qwen2VLForConditionalGeneration.from_pretrained(    MODEL_NAME,    quantization_config=quantization_settings,    torch_dtype=compute_dtype,    attn_implementation="sdpa",     # NOT flash_attention_2 - a T4 is too old for that    device_map="auto",)# The processor turns images and text into numbers the model understands.# max_pixels controls image resolution. Bigger = model can read smaller text,# but uses more memory. 1024*28*28 is a reasonable middle setting for documents.processor = AutoProcessor.from_pretrained(    MODEL_NAME,    min_pixels=256 * 28 * 28,    max_pixels=1024 * 28 * 28,)print("Model loaded")

## 8. Test the model BEFORE training**Do not skip this.** This is the number you compare against later.If you only measure after training, and you get 75%, you have no idea whether that is good.Maybe the model already did 75% before you touched it — and your training did nothing.

In [ ]:
LABEL_OPTIONS = ["Bill Summary", "Bill Detail", "Pharmacy Bill", "Lab Bill", "Other"]QUESTION = (    "Look at this medical bill page and tell me its type.\n"    "Choose exactly one from this list:\n"    + "\n".join(f"- {label}" for label in LABEL_OPTIONS)    + "\n\nAnswer with only the type name, nothing else.")from PIL import Imagedef ask_model_one_page(image_path):    """Show one page to the model and get back a label."""    image = Image.open(image_path).convert("RGB")    conversation = [{        "role": "user",        "content": [{"type": "image"}, {"type": "text", "text": QUESTION}],    }]    prompt_text = processor.apply_chat_template(        conversation, tokenize=False, add_generation_prompt=True    )    model_inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")    model_inputs = model_inputs.to(model.device)    with torch.no_grad():        output_ids = model.generate(**model_inputs, max_new_tokens=12, do_sample=False)    # Remove the prompt part, keep only what the model generated    generated_ids = output_ids[:, model_inputs.input_ids.shape[1]:]    answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]    # The model might write something slightly different, e.g. "bill detail".    # Match it to one of our five labels.    answer_lower = answer.strip().lower()    for label in LABEL_OPTIONS:        if label.lower() in answer_lower:            return label    return "Other"          # could not understand the answerdef measure_accuracy(pages, description):    """Ask the model about every page and count how many it got right."""    correct = 0    predictions = []    for page in pages:        predicted = ask_model_one_page(page["image_path"])        predictions.append(predicted)        if predicted == page["label"]:            correct += 1    accuracy = correct / len(pages)    print(f"{description}: {correct} correct out of {len(pages)} = {accuracy:.0%}")    return accuracy, predictionsprint("Testing the model BEFORE any training. This takes a few minutes...\n")accuracy_before, predictions_before = measure_accuracy(test_pages, "BEFORE training")

## 9. Prepare the training dataWe turn each labelled page into a small conversation:- **user:** [the page image] + our question- **assistant:** the correct labelThe model learns by trying to produce the assistant's reply.

In [ ]:
def make_training_example(page):    return {        "image_path": page["image_path"],        "question": QUESTION,        "answer": page["label"],    }training_examples = [make_training_example(p) for p in train_pages]print(f"Prepared {len(training_examples)} training examples")print()print("One example looks like this:")print("  image :", training_examples[0]["image_path"])print("  answer:", training_examples[0]["answer"])

## 10. Set up LoRAQwen2-VL-2B has about 2 billion numbers inside it. Training all of them needs far more memorythan a T4 has.**LoRA** solves this. Instead of changing the original numbers, we freeze them and add a smallset of new numbers alongside — usually less than 1% of the total. We train only those.This is why it fits on a free GPU, and it is why the saved result is a ~40 MB file instead ofa 4 GB one.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training# Prepare a 4-bit model for training (handles some numerical details for us)model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)model.enable_input_require_grads()lora_settings = LoraConfig(    r=16,                    # size of the small added matrices. Bigger = more capacity + more memory    lora_alpha=32,           # a scaling factor, usually 2x r    lora_dropout=0.05,    bias="none",    task_type="CAUSAL_LM",    # Which parts of the model to add LoRA to. These names are the attention and    # feed-forward layers inside the language part of the model.    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",                    "gate_proj", "up_proj", "down_proj"],)model = get_peft_model(model, lora_settings)# This prints how few numbers we are actually trainingmodel.print_trainable_parameters()

## 11. TrainA few things to understand in the settings below:- **batch size 1 + accumulation 4** — we process one page at a time because that is all the  memory allows, but we wait for 4 pages before updating the model. The effect is like a  batch of 4.- **learning rate 1e-4** — how big a step to take each update. Too high and training becomes  unstable, too low and nothing changes.- **fp16** — because a T4 cannot do bf16.With about 35 pages this takes only a few minutes.

In [ ]:
from torch.utils.data import Datasetfrom transformers import Trainer, TrainingArgumentsclass PageDataset(Dataset):    def __init__(self, examples):        self.examples = examples    def __len__(self):        return len(self.examples)    def __getitem__(self, i):        return self.examples[i]def collate_batch(batch):    """Turn a list of examples into tensors the model can train on."""    images, full_texts, prompt_lengths = [], [], []    for example in batch:        image = Image.open(example["image_path"]).convert("RGB")        images.append(image)        # The full conversation: question AND answer        full_conversation = [            {"role": "user", "content": [{"type": "image"},                                         {"type": "text", "text": example["question"]}]},            {"role": "assistant", "content": [{"type": "text", "text": example["answer"]}]},        ]        full_texts.append(            processor.apply_chat_template(full_conversation, tokenize=False)        )        # Just the question part, so we know where the answer begins        question_only = [full_conversation[0]]        question_text = processor.apply_chat_template(            question_only, tokenize=False, add_generation_prompt=True        )        question_tokens = processor(text=[question_text], images=[image],                                    return_tensors="pt").input_ids        prompt_lengths.append(question_tokens.shape[1])    inputs = processor(text=full_texts, images=images,                       return_tensors="pt", padding=True)    # "labels" tells the model what it should have produced.    # We set the question part to -100, which means "do not learn from this part".    # We only want the model to learn to produce the ANSWER, not to repeat our question.    labels = inputs.input_ids.clone()    labels[inputs.attention_mask == 0] = -100        # ignore padding    for i, length in enumerate(prompt_lengths):        labels[i, :length] = -100                    # ignore the question    inputs["labels"] = labels    return inputsprocessor.tokenizer.padding_side = "right"training_settings = TrainingArguments(    output_dir="my_adapter",    num_train_epochs=8,              # how many times to go through the data    per_device_train_batch_size=1,    gradient_accumulation_steps=4,    learning_rate=1e-4,    lr_scheduler_type="cosine",    warmup_ratio=0.05,    logging_steps=5,    save_strategy="no",    fp16=not use_bfloat16,    bf16=use_bfloat16,    gradient_checkpointing=True,    gradient_checkpointing_kwargs={"use_reentrant": False},    remove_unused_columns=False,     # our collator needs the raw dictionaries    report_to=[],)trainer = Trainer(    model=model,    args=training_settings,    train_dataset=PageDataset(training_examples),    data_collator=collate_batch,)print("Training now. Watch the 'loss' number - it should go down.\n")trainer.train()print("\nTraining finished")

## 12. Test again and compareThis is the moment of truth. Three numbers matter:1. **Always-guess baseline** — what you get by guessing the most common label2. **Before training** — what the model could already do3. **After training** — what it can do nowYour training only helped if number 3 is clearly above both 1 and 2.

In [ ]:
model.eval()print("Testing the model AFTER training...\n")accuracy_after, predictions_after = measure_accuracy(test_pages, "AFTER training")print("\n" + "=" * 50)print("RESULTS")print("=" * 50)print(f"Always guess most common label : {baseline_accuracy:.0%}")print(f"Model before training          : {accuracy_before:.0%}")print(f"Model after training           : {accuracy_after:.0%}")print()if accuracy_after > accuracy_before and accuracy_after > baseline_accuracy:    print("Training helped. The model is better than both the baseline and its starting point.")elif accuracy_after <= baseline_accuracy:    print("Training did NOT beat the always-guess baseline.")    print("This is a real result, not a failure to hide. Likely causes:")    print("  - too little data (35 pages is very few)")    print("  - the labels are ambiguous (some pages genuinely fit two categories)")    print("Report it honestly. Knowing your model did not work is worth more")    print("than a number you cannot explain.")else:    print("Training did not improve on the starting point. Try more epochs,")    print("or check whether your labels are consistent.")print("\nPages the model got wrong after training:")for page, predicted in zip(test_pages, predictions_after):    if predicted != page["label"]:        print(f"  {page['document']} page {page['page_number']}: "              f"said '{predicted}', should be '{page['label']}'")

## 13. Save your adapterThe adapter is small — about 40 MB. The big base model stays public on Hugging Face,so this file is all you need to keep.

In [ ]:
model.save_pretrained("my_adapter")!du -sh my_adapter!tar czf my_adapter.tar.gz my_adapterfrom google.colab import filesfiles.download("my_adapter.tar.gz")

## What to say about this in an interviewDo not say *"I fine-tuned a VLM and it works."* Say what you actually measured:> "I fine-tuned Qwen2-VL-2B with LoRA to classify medical bill pages into five types.> I hand-labelled 50 pages myself rather than using another model to label them, because> otherwise I would only be measuring agreement with that model, not accuracy.>> I split by document rather than by page, since pages from the same bill share a template> and splitting by page would leak information into the test set.>> I measured three things: an always-guess-the-most-common-class baseline, the base model> before any training, and the model after training. [Then give your three numbers.]>> The test set is only about 15 pages, so the confidence interval is wide. I would call this> a pilot result, not a reliable accuracy figure."That last paragraph is the one that will impress them. Most candidates quote a number.Very few say how much to trust it.### If the result was badSay so. *"Training did not beat the baseline, and I think it is because 35 training pagesis not enough for five classes"* is a stronger answer than a number you cannot defend.The JD says **measure everything** — it does not say every experiment must succeed.